# BTC Price Predictor — `pridictior_snow9` (Snowflake Notebook)

Runs the **full pipeline offline** on the Snowflake container runtime
(4 × A10 23 GB, 48 vCPU, 100 GB RAM):

1. creates the database/stage if missing, then pulls the dataset and the code
   from the internal stage `@SNOW9.PUBLIC.BTC_SNOW9_STAGE`
   (**no internet required** — only PyPI for packages, which is allowed)
2. builds the Market-Analyser feature cache (CPU)
3. trains **four Price-Predictor variants, one per GPU**, with the
   streaming *real-time reward* (no epochs, 25-min-ahead, updated every second)
4. replays a **random market day (2020 → today)**, rendering the 30 fps
   animation: actual price, dotted 25-min forecasts per model, live accuracy bars


In [ ]:
# ---- 1. packages (PyPI is reachable; everything else is offline) ----
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch", "--index-url", "https://download.pytorch.org/whl/cu121"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy", "pandas", "pyarrow", "matplotlib",
                "pyyaml", "tqdm", "imageio[ffmpeg]"])
print("packages ok")


In [ ]:
# ---- 2. create DB/stage if missing, pull code + data (NO internet) ----
# tolerant: finds ANY .zip in the stage (e.g. "pridictior_snow9.zip" or
# GitHub's "pridictior_snow9-main.zip"), extracts it wherever it unpacks to,
# and mirrors the folder that contains src/snow9 into /tmp/snow9/repo
import os, glob, zipfile, shutil
from snowflake.snowpark.context import get_active_session
session = get_active_session()

session.sql("CREATE DATABASE IF NOT EXISTS SNOW9").collect()
session.sql("""CREATE STAGE IF NOT EXISTS SNOW9.PUBLIC.BTC_SNOW9_STAGE
    DIRECTORY = (ENABLE = TRUE)""").collect()

stage_files = session.sql("LIST @SNOW9.PUBLIC.BTC_SNOW9_STAGE").to_pandas()
zips = [r.name for r in stage_files.itertuples() if str(r.name).endswith(".zip")]
pqs  = [r.name for r in stage_files.itertuples() if str(r.name).endswith(".parquet")]
assert zips, ("No .zip found in @SNOW9.PUBLIC.BTC_SNOW9_STAGE.\n"
              "  zip the repo (or GitHub: Code -> Download ZIP), then upload:\n"
              "  quick : Snowsight -> Data -> SNOW9 -> PUBLIC -> Stages ->\n"
              "          BTC_SNOW9_STAGE -> '+ Files'\n"
              "  bulk  : SnowSQL/CLI, see snowflake/upload_data_to_stage.sql")
assert pqs, ("No .parquet shards in stage. Upload data/raw/*.parquet "
             "(scripts/01_download_data.sh on an internet machine first).")

DATA = "/tmp/snow9/data/raw"; os.makedirs(DATA, exist_ok=True)
for p in pqs:
    session.file.get(p if str(p).startswith("@") else "@" + p, DATA + "/")
for z in zips:
    session.file.get(z if str(z).startswith("@") else "@" + z, "/tmp/snow9/")

os.makedirs("/tmp/snow9/extracted", exist_ok=True)
for local in glob.glob("/tmp/snow9/**/*.zip", recursive=True):
    zipfile.ZipFile(local).extractall("/tmp/snow9/extracted")

repo_root = next((root for root, d, f in os.walk("/tmp/snow9/extracted")
                  if os.path.isdir(os.path.join(root, "src", "snow9"))), None)
assert repo_root, "zip found but it has no src/snow9 folder — wrong archive?"
shutil.copytree(repo_root, "/tmp/snow9/repo", dirs_exist_ok=True)
print("repo ready:", os.listdir("/tmp/snow9/repo"),
      "| parquet shards:", len(os.listdir(DATA)))


In [ ]:
# ---- 3. feature cache (single vectorised pass over every shard) ----
import sys, os
sys.path.insert(0, "/tmp/snow9/repo/src"); os.chdir("/tmp/snow9/repo")
from snow9.data import ShardStore, build_feature_cache
from snow9.models import MarketAnalyzer
meta = build_feature_cache(ShardStore("/tmp/snow9/data/raw"),
                           "/tmp/snow9/data/cache", MarketAnalyzer())
meta


In [ ]:
# ---- 4. TRAIN: 4 variants x 1 GPU each, streaming real-time reward ----
import torch
assert torch.cuda.device_count() == 4, "expected the 4 x A10 node"
for i in range(4): print(i, torch.cuda.get_device_name(i))

import subprocess, os
env = dict(os.environ, PYTHONPATH="/tmp/snow9/repo/src", OMP_NUM_THREADS="10")
subprocess.run(
    "torchrun --standalone --nproc_per_node=4 "
    "-m snow9.training.ddp_launch "
    "--data-cache /tmp/snow9/data/cache "
    "--variants configs/variants.yaml "
    "--training configs/training.yaml "
    "--out /tmp/snow9/checkpoints",
    shell=True, cwd="/tmp/snow9/repo", env=env, check=True)
# full snapshots every 30 market-minutes per variant — storage is not a constraint


In [ ]:
# ---- 5. REPLAY a random market day through ALL trained variants ----
import subprocess, os, sys
env = dict(os.environ, PYTHONPATH="/tmp/snow9/repo/src")
subprocess.run([sys.executable, "-m", "snow9.viz.rollout",
                "--cache", "/tmp/snow9/data/cache",
                "--ckpt-root", "/tmp/snow9/checkpoints",
                "--date", "random", "--duration-sec", "86400",
                "--out", "/tmp/snow9/runs/rollout_random.npz"],
               cwd="/tmp/snow9/repo", env=env, check=True)


In [ ]:
# ---- 6. RENDER the 30 fps animation and watch it here ----
import subprocess, os, sys
env = dict(os.environ, PYTHONPATH="/tmp/snow9/repo/src")
subprocess.run([sys.executable, "-m", "snow9.viz.animate",
                "/tmp/snow9/runs/rollout_random.npz",
                "--out", "/tmp/snow9/videos/predictions_random.mp4",
                "--fps", "30", "--seconds-per-frame", "120"],
               cwd="/tmp/snow9/repo", env=env, check=True)
from IPython.display import Video
Video("/tmp/snow9/videos/predictions_random.mp4", embed=True, width=1000)


### Notes
* Checkpoints (model + optimiser + EMA + RNG, every 30 market-min per variant) live in
  `/tmp/snow9/checkpoints/<variant>/` — persist them back to the stage with
  `session.file.put` if you want them to outlive this warehouse.
* Training resumes automatically from each variant's `latest.pt`.
* Not financial advice — see `README.md`.
